# 05. Détection temporelle et métriques par événement

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # racine du projet
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, io, viz, models, features
from src import postprocess as pp, evaluation as ev
pd.set_option("display.width", 200)
print("Dossier données :", config.DATA_DIR)

Objectif : transformer les probabilités par fenêtre en **événements** (lissage, seuil, hystérésis), puis mesurer détection, fausses alarmes/heure et latence.

In [ ]:
preds = pd.read_parquet(config.CACHE_DIR / "predictions.parquet")
pid = sorted(preds.patient.unique())[0]
g = preds[preds.patient == pid].sort_values("t_start")
sm, intervals = pp.postprocess(g.t_start.values, g.score.values,
                               threshold=0.4, smooth_size=5, min_gap_s=2, min_dur_s=2)
print("Intervalles détectés :", intervals)

In [ ]:
em = pd.read_parquet(config.CACHE_DIR / "event_metrics.parquet")
em.describe().round(2)

In [ ]:
sweep = pd.read_parquet(config.CACHE_DIR / "threshold_sweep.parquet")
fig, ax = plt.subplots(figsize=(8,4))
for c in ["precision","recall","f1","mcc"]:
    ax.plot(sweep.threshold, sweep[c], label=c)
ax.set_xlabel("Seuil"); ax.legend(); plt.show()

Courbe seuils événementiels (étape 5b) : à seuil 0,2 on détecte 86,6 % des crises pour 3,35 FA/h.